# 🌀 SIFE-LDM V3.1 — Training on Google Colab (GPU)

**Structured Intelligence Field Equation — Latent Diffusion Model**

This notebook supports **three training modes**:

| Mode | Dataset | What it learns |
|---|---|---|
| **Vision** | CIFAR-100 (50k images, 100 classes) | Class-conditional image generation |
| **Text** | MBPP / custom text / code files | Complex-field text/code generation |
| **Both** | Alternating text + vision batches | Dual-mode generation |

**Architecture:** Shape-Invariant Transformer + Patch Encoder + Phase-Coherent CFG  
**Stability:** Resolved `ScopeParamShapeError` for multi-resolution training via Sinusoidal positional embeddings and invariant pooling.  
**Optimizer:** ANDI (WSD schedule)  
**Physics:** SIFE field dynamics with phase coupling + stability regularization

---
## 1️⃣ Environment Setup

In [ ]:
import os

# === JAX Environment Variables (MUST be set before importing JAX) ===
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'
os.environ['XLA_PYTHON_CLIENT_ALLOCATOR'] = 'platform'
os.environ['JAX_TRACEBACK_FILTERING'] = 'off'

import jax
print(f'JAX version:  {jax.__version__}')
print(f'Devices:      {jax.devices()}')
print(f'Backend:      {jax.default_backend()}')
if jax.default_backend() != 'gpu':
    print('[WARN] No GPU! Go to: Runtime > Change runtime type > GPU (T4)')
else:
    print(f'[OK] GPU ready: {jax.devices()[0].device_kind}')


In [ ]:
# Install dependencies (Colab has JAX pre-installed)
!pip install -q flax optax datasets transformers
print('\u2705 Dependencies installed')


In [ ]:
# Upload your project code
# Upload the sife-ldm-colab.zip file

import os, sys, glob, shutil
WORK_DIR = '/content/sife-ldm'

# === FORCE CLEAN INSTALL ===
if os.path.exists(WORK_DIR):
    print('[CLEAN] Removing old installation...')
    shutil.rmtree(WORK_DIR)
if os.path.exists('/content/_tmp'):
    shutil.rmtree('/content/_tmp')

# Purge any cached sife modules from previous runs
mods_to_remove = [m for m in sys.modules if m.startswith('sife')]
for m in mods_to_remove:
    del sys.modules[m]
print(f'[CLEAN] Purged {len(mods_to_remove)} cached modules')

from google.colab import files
print('Upload sife-ldm-colab.zip:')
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]

# Extract with Python zipfile (handles backslash paths from Windows)
import zipfile
os.makedirs(WORK_DIR, exist_ok=True)
with zipfile.ZipFile(zip_name, 'r') as zf:
    zf.extractall(WORK_DIR)
print(f'Extracted {zip_name} to {WORK_DIR}')

# Clean up __pycache__
for root, dirs, _f in os.walk(WORK_DIR):
    for d in dirs:
        if d == '__pycache__':
            shutil.rmtree(os.path.join(root, d))

# === SET ENVIRONMENT ===
os.chdir(WORK_DIR)
os.environ['PYTHONPATH'] = WORK_DIR
if WORK_DIR not in sys.path:
    sys.path.insert(0, WORK_DIR)
if '.' not in sys.path:
    sys.path.insert(0, '.')

print(f'Working directory: {os.getcwd()}')
print(f'Contents: {sorted(os.listdir("."))}')

# === VERIFY FIXES ===
with open('sife/unet.py', 'r') as f:
    content = f.read()
checks = [
    ('ComplexConv1D(features=self.features, kernel_size=3, strides=2)', 'PhasePool shape-invariant pooling'),
    ('SinusoidalPositionEmbedding', 'Resolution-independent positional embeddings'),
]
for pattern, desc in checks:
    status = 'OK' if pattern in content else 'MISSING'
    print(f'  [{status}] {desc}')


In [ ]:
# Verify all module imports work
import os, sys

# Ensure we're in the right directory
if not os.path.exists('sife/model.py'):
    os.chdir('/content/sife-ldm')
    if '.' not in sys.path:
        sys.path.insert(0, '.')

from sife.model import SIFELDM, SIFELDMConfig, create_train_state, train_step, save_checkpoint, create_optimizer
from sife.field import SIFEConfig
from sife.diffusion import DiffusionConfig, GaussianDiffusion, DDIMSampler
from sife.multiscale import create_multiscale_config
from sife.tokenizer import Vocabulary, SIFETokenizer, create_training_data
print('[OK] All imports successful')


---
## 2️⃣ Choose Training Mode

Set `TRAINING_MODE` to one of: `'vision'`, `'text'`, or `'both'`

In [ ]:
#@title Training Configuration { display-mode: "form" }

#@markdown ### Mode
TRAINING_MODE = 'vision'  #@param ['vision', 'text', 'both']

#@markdown ### Hyperparameters
BATCH_SIZE    = 16    #@param {type:"integer"}
EMBED_DIM     = 128   #@param {type:"integer"}
LEARNING_RATE = 2e-4  #@param {type:"number"}
MAX_STEPS     = 50000 #@param {type:"integer"}
SAVE_EVERY    = 5000  #@param {type:"integer"}
LOG_EVERY     = 100   #@param {type:"integer"}

#@markdown ### Text-specific (only used if mode is 'text' or 'both')
TEXT_DATA_SOURCE = 'mbpp'  #@param ['mbpp', 'custom_file', 'huggingface']
CUSTOM_TEXT_FILE = ''   #@param {type:"string"}
HF_DATASET_NAME = 'wikitext' #@param {type:"string"}
MAX_SEQ_LEN     = 256  #@param {type:"integer"}

# === Auto-scale based on GPU type ===
import jax
try:
    gpu_kind = jax.devices()[0].device_kind
    if 'A100' in gpu_kind:
        BATCH_SIZE, EMBED_DIM = 64, 256
        print(f'[AUTO] A100 detected -- scaled: batch={BATCH_SIZE}, embed={EMBED_DIM}')
    elif 'V100' in gpu_kind:
        BATCH_SIZE, EMBED_DIM = 32, 192
        print(f'[AUTO] V100 detected -- scaled: batch={BATCH_SIZE}, embed={EMBED_DIM}')
    elif 'L4' in gpu_kind:
        BATCH_SIZE, EMBED_DIM = 24, 128
        print(f'[AUTO] L4 detected -- scaled: batch={BATCH_SIZE}, embed={EMBED_DIM}')
except:
    pass

print(f'[OK] Mode: {TRAINING_MODE}')
print(f'     Batch: {BATCH_SIZE} | Embed: {EMBED_DIM} | LR: {LEARNING_RATE} | Steps: {MAX_STEPS}')
if TRAINING_MODE in ('text', 'both'):
    print(f'     Text source: {TEXT_DATA_SOURCE} | Seq len: {MAX_SEQ_LEN}')


---
## 3️⃣ Prepare Data

In [ ]:
import numpy as np
import jax.numpy as jnp

images, labels = None, None
NUM_CLASSES = 0

if TRAINING_MODE in ('vision', 'both'):
    DATA_DIR = './data/cifar100'
    os.makedirs(DATA_DIR, exist_ok=True)

    if not os.path.exists(os.path.join(DATA_DIR, 'train_images.npy')):
        print('Downloading CIFAR-100...')
        from datasets import load_dataset
        ds = load_dataset('cifar100', split='train')
        images = np.array([np.array(img['img']) for img in ds])
        labels = np.array([img['fine_label'] for img in ds])
        np.save(os.path.join(DATA_DIR, 'train_images.npy'), images)
        np.save(os.path.join(DATA_DIR, 'train_labels.npy'), labels)
        print(f'\u2705 Downloaded: {images.shape} images, {labels.shape} labels')
    else:
        images = np.load(os.path.join(DATA_DIR, 'train_images.npy'))
        labels = np.load(os.path.join(DATA_DIR, 'train_labels.npy')).flatten()
        print(f'\u2705 CIFAR-100 loaded: {images.shape}')

    NUM_CLASSES = int(labels.max()) + 1
    print(f'   {len(images)} images, {NUM_CLASSES} classes, range=[{images.min()}, {images.max()}]')
else:
    print('\u23e9 Skipping vision data (text-only mode)')

In [ ]:
text_train_data = None
vocab = None
tokenizer = None
text_dataset = None

if TRAINING_MODE in ('text', 'both'):
    # --- Load text data ---
    if TEXT_DATA_SOURCE == 'mbpp':
        # Use the included MBPP code dataset
        mbpp_path = './data/processed_mbpp/train.txt'
        vocab_path = './data/processed_mbpp/vocab.json'
        if os.path.exists(mbpp_path):
            with open(mbpp_path, 'r', encoding='utf-8') as f:
                text_train_data = [line.strip() for line in f if line.strip()]
            print(f'\u2705 MBPP data loaded: {len(text_train_data)} samples')
        else:
            print('\u26a0\ufe0f MBPP data not found, downloading...')
            from datasets import load_dataset
            ds = load_dataset('mbpp', split='train')
            text_train_data = [item['code'] for item in ds]
            os.makedirs('./data/processed_mbpp', exist_ok=True)
            with open(mbpp_path, 'w', encoding='utf-8') as f:
                for t in text_train_data:
                    # Collapse to single line for simple line-based loading
                    f.write(t.replace('\n', ' \\n ') + '\n')
            print(f'\u2705 MBPP downloaded: {len(text_train_data)} samples')
            vocab_path = None

    elif TEXT_DATA_SOURCE == 'huggingface':
        from datasets import load_dataset
        print(f'Downloading {HF_DATASET_NAME}...')
        ds = load_dataset(HF_DATASET_NAME, split='train', streaming=True)
        text_train_data = []
        for i, item in enumerate(ds):
            if i >= 10000: break
            text_key = 'text' if 'text' in item else list(item.keys())[0]
            if item[text_key].strip():
                text_train_data.append(item[text_key].strip())
        print(f'\u2705 {HF_DATASET_NAME}: {len(text_train_data)} samples')
        vocab_path = None

    elif TEXT_DATA_SOURCE == 'custom_file':
        assert CUSTOM_TEXT_FILE, 'Set CUSTOM_TEXT_FILE path above'
        with open(CUSTOM_TEXT_FILE, 'r', encoding='utf-8') as f:
            text_train_data = [line.strip() for line in f if line.strip()]
        print(f'\u2705 Custom file: {len(text_train_data)} samples')
        vocab_path = None

    # --- Build or load vocabulary ---
    if vocab_path and os.path.exists(vocab_path):
        vocab = Vocabulary.load(vocab_path)
        print(f'   Vocabulary loaded: {len(vocab)} tokens')
    else:
        vocab = Vocabulary(min_freq=2, max_size=32000)
        vocab.build_from_texts(text_train_data[:10000])
        os.makedirs('./output', exist_ok=True)
        vocab.save('./output/vocab.json')
        print(f'   Vocabulary built: {len(vocab)} tokens (saved to output/vocab.json)')

    # --- Create tokenizer and dataset ---
    import jax
    tokenizer = SIFETokenizer(vocab=vocab, embed_dim=EMBED_DIM, max_seq_len=MAX_SEQ_LEN)

    # Save text data to a temp file for create_training_data
    _text_path = './data/_colab_text_train.txt'
    with open(_text_path, 'w', encoding='utf-8') as f:
        for t in text_train_data:
            f.write(t + '\n')

    text_dataset, _ = create_training_data(
        tokenizer, text_path=_text_path, batch_size=BATCH_SIZE
    )
    print(f'   Dataset: {len(text_dataset)} batches per epoch')

else:
    print('\u23e9 Skipping text data (vision-only mode)')

---
## 4️⃣ Initialize Model

In [ ]:
import jax
import os

# Ensure working directory
if not os.path.exists('sife/model.py'):
    os.chdir('/content/sife-ldm')

sife_config = SIFEConfig(k=1.0, dt=0.005, gamma=0.1, eta_damping=0.05)
ms_config   = create_multiscale_config(num_levels=3, base_features=64)
diff_config = DiffusionConfig(num_timesteps=1000)

config = SIFELDMConfig(
    sife=sife_config,
    diffusion=diff_config,
    multiscale=ms_config,
    model_type=TRAINING_MODE,
    is_image=(TRAINING_MODE in ('vision', 'both')),
    image_size=(32, 32),
    embed_dim=EMBED_DIM,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    max_steps=MAX_STEPS,
    num_classes=NUM_CLASSES,
    max_seq_len=MAX_SEQ_LEN,
)

print(f'Initializing SIFE-LDM ({TRAINING_MODE} mode)...')
key = jax.random.PRNGKey(42)
model, state, diffusion = create_train_state(config, key)
optimizer = create_optimizer(config)

param_count = sum(x.size for x in jax.tree_util.tree_leaves(state.params))
print(f'[OK] Model ready: {param_count:,} parameters (~{param_count * 4 / 1e9:.2f} GB)')


---
## 5️⃣ Train 🚀

Runs end-to-end training with checkpointing.

In [ ]:
import time

os.makedirs('./checkpoints', exist_ok=True)

loss_history = []
text_iter = iter(text_dataset) if text_dataset else None

print(f'Starting {TRAINING_MODE} training for {MAX_STEPS} steps on {jax.default_backend().upper()}...')
print('=' * 70)
t0 = time.time()

for step in range(MAX_STEPS):

    # ── Prepare batch based on mode ──
    if TRAINING_MODE == 'vision':
        idx = np.random.randint(0, len(images), BATCH_SIZE)
        batch_img = jnp.array(images[idx].astype(np.float32) / 255.0)
        batch_lbl = jnp.array(labels[idx].astype(np.int32))
        key, subkey = jax.random.split(key)
        batch = {
            'images': batch_img,
            'labels': batch_lbl,
            'use_context_mask': jax.random.uniform(subkey, (BATCH_SIZE,)) > 0.15,
        }

    elif TRAINING_MODE == 'text':
        try:
            batch = next(text_iter)
        except StopIteration:
            text_iter = iter(text_dataset)
            batch = next(text_iter)

    else:  # both
        if step % 2 == 0:  # Vision batch on even steps
            idx = np.random.randint(0, len(images), BATCH_SIZE)
            batch_img = jnp.array(images[idx].astype(np.float32) / 255.0)
            batch_lbl = jnp.array(labels[idx].astype(np.int32))
            key, subkey = jax.random.split(key)
            batch = {
                'images': batch_img,
                'labels': batch_lbl,
                'use_context_mask': jax.random.uniform(subkey, (BATCH_SIZE,)) > 0.15,
            }
        else:  # Text batch on odd steps
            try:
                batch = next(text_iter)
            except StopIteration:
                text_iter = iter(text_dataset)
                batch = next(text_iter)

    # ── Train step ──
    state, metrics = train_step(model, state, batch, diffusion, optimizer)

    loss_val = float(metrics['loss'])
    loss_history.append(loss_val)

    if step % LOG_EVERY == 0:
        elapsed = time.time() - t0
        sps = (step + 1) / elapsed if elapsed > 0 else 0
        avg = np.mean(loss_history[-LOG_EVERY:]) if loss_history else loss_val
        eta = (MAX_STEPS - step) / max(sps, 1e-6) / 60
        tag = '\U0001f5bc' if (TRAINING_MODE == 'both' and step % 2 == 0) else '\U0001f4dd' if TRAINING_MODE != 'vision' else ''
        print(f'Step {step:>6d}/{MAX_STEPS} | Loss: {loss_val:.4f} | '
              f'Avg: {avg:.4f} | {sps:.2f} it/s | ETA: {eta:.0f}m {tag}')

    if step > 0 and step % SAVE_EVERY == 0:
        save_checkpoint(state, './checkpoints', step)

# Final save
save_checkpoint(state, './checkpoints', MAX_STEPS, name=f'final_{TRAINING_MODE}')
elapsed = time.time() - t0
print('=' * 70)
print(f'\u2705 Done! {MAX_STEPS} steps in {elapsed/60:.1f}min | Final loss: {loss_history[-1]:.4f}')

---
## 6️⃣ Training Curves

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(loss_history, alpha=0.3, color='steelblue', lw=0.5)
w = min(100, len(loss_history))
if len(loss_history) >= w:
    sm = np.convolve(loss_history, np.ones(w)/w, mode='valid')
    ax1.plot(range(w-1, len(loss_history)), sm, color='orangered', lw=2, label=f'{w}-step avg')
ax1.set(xlabel='Step', ylabel='Loss', title=f'SIFE-LDM Training Loss ({TRAINING_MODE})')
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.semilogy(loss_history, alpha=0.3, color='steelblue', lw=0.5)
if len(loss_history) >= w:
    ax2.semilogy(range(w-1, len(loss_history)), sm, color='orangered', lw=2)
ax2.set(xlabel='Step', ylabel='Loss (log)', title='Loss (log scale)')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_loss.png', dpi=150)
plt.show()
print(f'Loss range: {min(loss_history):.4f} \u2014 {max(loss_history):.4f}')

---
## 7A️⃣ Generate Images (vision / both mode)

In [ ]:
import matplotlib.pyplot as plt
if TRAINING_MODE in ('vision', 'both'):
    def generate_image(label_id, key, steps=50):
        H, W = config.image_size
        shape = (1, H, W, config.embed_dim)
        ddim = DDIMSampler(diffusion)
        lbl = jnp.array([label_id], dtype=jnp.int32)
        def model_fn(x, t, ctx):
            return model.apply(state.params, x, t, labels=ctx, deterministic=True)
        latent = ddim.sample(model_fn, shape, key, lbl, num_steps=steps)
        # The new V3.1 architecture is stable for direct decoding
        rgb = model.apply(state.params, latent, method=model.decode_images)
        return np.clip(np.array(rgb[0]), 0, 1)


---
## 7B️⃣ Generate Text (text / both mode)

In [ ]:
if TRAINING_MODE not in ('text', 'both'):
    print('\u23e9 Skipping text generation (vision-only mode)')
else:
    from sife.tokenizer import generate_text

    prompts = [
        'def fibonacci(',
        'def sort_list(',
        'import os',
    ]

    print('Generating text samples...')
    print('=' * 60)
    for prompt in prompts:
        key, sk = jax.random.split(key)
        try:
            output = generate_text(
                model=lambda x, t, ctx: model.apply(state.params, x, t, deterministic=True),
                tokenizer=tokenizer,
                prompt=prompt,
                diffusion=diffusion,
                key=sk,
                num_steps=50,
                max_length=MAX_SEQ_LEN,
            )
            print(f'\n\u250c\u2500 Prompt: {prompt}')
            print(f'\u2514\u2500 Output: {output}')
        except Exception as e:
            print(f'\n\u250c\u2500 Prompt: {prompt}')
            print(f'\u2514\u2500 Error: {e}')
    print('\n' + '=' * 60)

---
## 8️⃣ Download Results

In [ ]:
from google.colab import files

!zip -r sife_ldm_trained.zip checkpoints/ training_loss.png \
    $([ -f generated_images.png ] && echo generated_images.png) \
    $([ -f output/vocab.json ] && echo output/vocab.json)
files.download('sife_ldm_trained.zip')
print('\u2705 Download started')